In [1]:
import torch
from torch import nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

In [2]:
class TwoLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(TwoLayer, self).__init__()

        self.linear1 = nn.Linear(input_dim, 28)
        self.linear2 = nn.Linear(28,output_dim)

    def forward(self, x, **kwargs):
        feature = kwargs.get('need_features', False)

        x = F.relu(self.linear1(x))
        feat = x
        x = self.linear2(x)

        if feature:
            return x, feat
        else:
            return x
    
    def feat_nograd_forward(self, x):
        with torch.no_grad():
            x = F.relu(self.linear1(x))
            feat = torch.flatten(x, 1)
        x = self.linear2(feat)
        return x, feat

In [3]:
def init_weights(m):
  if isinstance(m, nn.Linear):
    nn.init.uniform_(m.weight, -1, 1)
    # nn.init.xavier_uniform_(m.weight)
    # nn.init.sparse_(m.weight, sparsity=0.6, std = 5.0)

# teacher_model = nn.Sequential(nn.Linear(16,28), nn.ReLU(), nn.Linear(28, 4))
teacher_model = TwoLayer(input_dim=16, output_dim=4)
teacher_model.apply(init_weights)

TwoLayer(
  (linear1): Linear(in_features=16, out_features=28, bias=True)
  (linear2): Linear(in_features=28, out_features=4, bias=True)
)

In [4]:
tau = 3

X = (torch.rand(1000,16) - 0.5)*2.0
with torch.no_grad():
  Y = teacher_model(X)
  # Y = Y + torch.randn(Y.shape)*0.1 # add error
Y = torch.softmax(Y / tau, dim=-1)
predicted_labels = torch.argmax(Y, dim=-1)

In [5]:
X = X.numpy()
Y = Y.numpy()
predicted_labels = predicted_labels.numpy()

In [6]:
for i in range(4):
  print(len(predicted_labels[predicted_labels == i]))

224
129
552
95


In [7]:
import os

# Define your save path
SAVE_PATH = "./models/teacher/teacher_generating.pth"

# Ensure the directory exists
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

# Save the weights
torch.save(teacher_model.state_dict(), SAVE_PATH)